# Config

> Configuration dataclasses and enums for the media gallery.

In [ ]:
#| default_exp core.config

In [ ]:
#| export
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, List, Literal, Optional, Tuple

from cjm_file_discovery.core.models import FileInfo, FileType

## Enums

Enumeration types for gallery configuration options.

In [ ]:
#| export
class ViewMode(str, Enum):
    """Display modes for media gallery."""
    GRID = "grid"   # Card grid layout
    LIST = "list"   # Table/list layout

In [ ]:
#| export
class SelectionMode(str, Enum):
    """File selection modes."""
    NONE = "none"           # Browse only, no selection
    SINGLE = "single"       # Select one file
    MULTIPLE = "multiple"   # Select multiple files

In [ ]:
#| export
class CardSize(str, Enum):
    """Card size presets for grid view."""
    SMALL = "small"     # Compact cards
    MEDIUM = "medium"   # Default size
    LARGE = "large"     # Large cards with more detail

In [ ]:
#| export
class ListColumn(str, Enum):
    """Available columns for list view."""
    NAME = "name"
    TYPE = "type"
    SIZE = "size"
    MODIFIED = "modified"
    EXTENSION = "extension"
    PATH = "path"

In [ ]:
# Test enums
assert ViewMode.GRID.value == "grid"
assert ViewMode.LIST == "list"
assert SelectionMode.SINGLE.value == "single"
assert CardSize.MEDIUM.value == "medium"
assert ListColumn.NAME.value == "name"
print("Enum tests passed!")

Enum tests passed!


## GridConfig

Configuration for the grid view layout.

In [ ]:
#| export
@dataclass
class GridConfig:
    """Grid view display configuration."""
    columns: int = 4                       # Number of columns in grid
    card_size: CardSize = CardSize.MEDIUM  # Card size preset
    show_thumbnails: bool = True           # Show image/video thumbnails
    show_file_size: bool = True            # Show file size on cards
    show_file_type: bool = True            # Show file type badge

In [ ]:
# Test GridConfig defaults
grid_config = GridConfig()
assert grid_config.columns == 4
assert grid_config.card_size == CardSize.MEDIUM
assert grid_config.show_thumbnails == True

# Test custom config
custom_grid = GridConfig(
    columns=6,
    card_size=CardSize.SMALL,
    show_thumbnails=False
)
assert custom_grid.columns == 6
assert custom_grid.card_size == CardSize.SMALL
assert custom_grid.show_thumbnails == False

print("GridConfig tests passed!")

GridConfig tests passed!


## ListConfig

Configuration for the list view layout.

In [ ]:
#| export
@dataclass
class ListConfig:
    """List view display configuration."""
    columns: List[ListColumn] = field(default_factory=lambda: [
        ListColumn.NAME, ListColumn.TYPE, ListColumn.SIZE, ListColumn.MODIFIED
    ])
    show_icons: bool = True                # Show file type icons
    striped: bool = False                  # Zebra striping for rows
    compact: bool = False                  # Compact row height

In [ ]:
# Test ListConfig defaults
list_config = ListConfig()
assert ListColumn.NAME in list_config.columns
assert ListColumn.SIZE in list_config.columns
assert list_config.show_icons == True
assert list_config.striped == False

# Test custom config
custom_list = ListConfig(
    columns=[ListColumn.NAME, ListColumn.TYPE],
    striped=True,
    compact=True
)
assert len(custom_list.columns) == 2
assert custom_list.striped == True
assert custom_list.compact == True

print("ListConfig tests passed!")

ListConfig tests passed!


## FilterConfig

Configuration for filtering displayed media files.

In [ ]:
#| export
@dataclass
class FilterConfig:
    """Filter configuration for media gallery."""
    enabled_types: List[FileType] = field(default_factory=lambda: [
        FileType.AUDIO, FileType.VIDEO, FileType.IMAGE, FileType.DOCUMENT
    ])  # File types to display
    allow_type_filter: bool = True         # Show type filter controls
    show_hidden: bool = False              # Show hidden files
    custom_filter: Optional[Callable[[FileInfo], bool]] = None  # Custom filter function
    
    def matches(
        self,
        file_info: FileInfo  # File to check against filter
    ) -> bool:  # True if file passes filter
        """Check if a file matches the filter criteria."""
        # Check hidden files
        if not self.show_hidden and file_info.name.startswith('.'):
            return False
        
        # Check file type (skip directories)
        if not file_info.is_directory:
            if file_info.file_type not in self.enabled_types:
                return False
        
        # Check custom filter
        if self.custom_filter is not None:
            return self.custom_filter(file_info)
        
        return True

In [ ]:
# Test FilterConfig
filter_config = FilterConfig()

# Test default (media types enabled)
audio_file = FileInfo(
    name="song.mp3", path="/song.mp3", is_directory=False,
    file_type=FileType.AUDIO
)
assert filter_config.matches(audio_file) == True

# Test hidden file filtering
hidden_file = FileInfo(
    name=".hidden.mp3", path="/.hidden.mp3", is_directory=False,
    file_type=FileType.AUDIO
)
assert filter_config.matches(hidden_file) == False

filter_with_hidden = FilterConfig(show_hidden=True)
assert filter_with_hidden.matches(hidden_file) == True

# Test type filtering
code_file = FileInfo(
    name="script.py", path="/script.py", is_directory=False,
    file_type=FileType.CODE
)
assert filter_config.matches(code_file) == False  # CODE not in default enabled

# Test enabled types
audio_only_filter = FilterConfig(enabled_types=[FileType.AUDIO])
video_file = FileInfo(
    name="video.mp4", path="/video.mp4", is_directory=False,
    file_type=FileType.VIDEO
)
assert audio_only_filter.matches(audio_file) == True
assert audio_only_filter.matches(video_file) == False

# Test custom filter
def size_filter(f: FileInfo) -> bool:
    return f.size is None or f.size < 1000000  # Under 1MB

custom_filter = FilterConfig(custom_filter=size_filter)
small_file = FileInfo(
    name="small.mp3", path="/small.mp3", is_directory=False,
    file_type=FileType.AUDIO, size=500
)
large_file = FileInfo(
    name="large.mp3", path="/large.mp3", is_directory=False,
    file_type=FileType.AUDIO, size=10000000
)
assert custom_filter.matches(small_file) == True
assert custom_filter.matches(large_file) == False

print("FilterConfig tests passed!")

FilterConfig tests passed!


## PaginationConfig

Configuration for pagination controls.

In [ ]:
#| export
@dataclass
class PaginationConfig:
    """Pagination configuration."""
    items_per_page: int = 24               # Items to show per page
    show_pagination: bool = True           # Show pagination controls
    show_page_size_selector: bool = False  # Allow changing page size
    page_size_options: List[int] = field(default_factory=lambda: [12, 24, 48, 96])

In [ ]:
# Test PaginationConfig
pagination = PaginationConfig()
assert pagination.items_per_page == 24
assert pagination.show_pagination == True
assert 24 in pagination.page_size_options

custom_pagination = PaginationConfig(
    items_per_page=48,
    show_page_size_selector=True
)
assert custom_pagination.items_per_page == 48
assert custom_pagination.show_page_size_selector == True

print("PaginationConfig tests passed!")

PaginationConfig tests passed!


## PreviewConfig

Configuration for the preview modal.

In [ ]:
#| export
@dataclass
class PreviewConfig:
    """Preview modal configuration."""
    enable_preview: bool = True            # Enable preview modal on click
    show_download_button: bool = True      # Show download button in preview
    show_info_panel: bool = True           # Show file info in preview
    show_navigation: bool = True           # Show prev/next navigation
    autoplay_video: bool = False           # Autoplay videos in preview
    autoplay_audio: bool = False           # Autoplay audio in preview

In [ ]:
# Test PreviewConfig
preview = PreviewConfig()
assert preview.enable_preview == True
assert preview.show_download_button == True
assert preview.autoplay_video == False

custom_preview = PreviewConfig(
    show_navigation=False,
    autoplay_video=True
)
assert custom_preview.show_navigation == False
assert custom_preview.autoplay_video == True

print("PreviewConfig tests passed!")

PreviewConfig tests passed!


## GalleryCallbacks

Optional callbacks for customizing gallery behavior.

In [ ]:
#| export
@dataclass
class GalleryCallbacks:
    """Event callbacks for gallery customization."""
    on_select: Optional[Callable[[str], None]] = None            # Called when file selected
    on_selection_change: Optional[Callable[[List[str]], None]] = None  # Called when selection changes
    on_preview: Optional[Callable[[str], None]] = None           # Called when preview opens
    on_download: Optional[Callable[[str], None]] = None          # Called when download clicked
    validate_selection: Optional[Callable[[str], Tuple[bool, str]]] = None  # Validate before select

In [ ]:
# Test GalleryCallbacks
selected_files = []

def track_select(path: str):
    selected_files.append(path)

def validate_audio_only(path: str) -> Tuple[bool, str]:
    if path.endswith('.mp3') or path.endswith('.wav'):
        return (True, "")
    return (False, "Only audio files allowed")

callbacks = GalleryCallbacks(
    on_select=track_select,
    validate_selection=validate_audio_only
)

# Test callback works
callbacks.on_select("/test/song.mp3")
assert selected_files == ["/test/song.mp3"]

# Test validation
valid, msg = callbacks.validate_selection("/test.mp3")
assert valid == True
valid, msg = callbacks.validate_selection("/test.mp4")
assert valid == False
assert "Only audio" in msg

print("GalleryCallbacks tests passed!")

GalleryCallbacks tests passed!


## GalleryConfig

Main configuration for the media gallery component.

In [ ]:
#| export
@dataclass
class GalleryConfig:
    """Main configuration for media gallery."""
    # View settings
    default_view: ViewMode = ViewMode.GRID     # Default view mode
    allow_view_toggle: bool = True             # Allow user to toggle view mode
    
    # Sub-configurations
    grid: GridConfig = field(default_factory=GridConfig)
    list: ListConfig = field(default_factory=ListConfig)
    filter: FilterConfig = field(default_factory=FilterConfig)
    pagination: PaginationConfig = field(default_factory=PaginationConfig)
    preview: PreviewConfig = field(default_factory=PreviewConfig)
    
    # Selection
    selection_mode: SelectionMode = SelectionMode.NONE
    max_selections: Optional[int] = None       # For MULTIPLE mode
    
    # HTML IDs (for HTMX targeting)
    gallery_id: str = "media-gallery"
    content_id: str = "media-gallery-content"
    controls_id: str = "media-gallery-controls"
    preview_modal_id: str = "media-preview-modal"
    
    # Keyboard (placeholder for future integration)
    keyboard_config: Optional[Any] = None
    
    def can_select(
        self,
        file_info: FileInfo  # File to check
    ) -> bool:  # True if file can be selected
        """Check if a file can be selected based on config."""
        if self.selection_mode == SelectionMode.NONE:
            return False
        # Directories are not selectable in gallery
        if file_info.is_directory:
            return False
        return True

In [ ]:
# Test GalleryConfig
config = GalleryConfig()

# Test defaults
assert config.default_view == ViewMode.GRID
assert config.allow_view_toggle == True
assert config.selection_mode == SelectionMode.NONE
assert config.gallery_id == "media-gallery"
assert isinstance(config.grid, GridConfig)
assert isinstance(config.list, ListConfig)
assert isinstance(config.filter, FilterConfig)
assert isinstance(config.pagination, PaginationConfig)
assert isinstance(config.preview, PreviewConfig)

# Test can_select
file = FileInfo(name="test.mp3", path="/test.mp3", is_directory=False, file_type=FileType.AUDIO)
folder = FileInfo(name="folder", path="/folder", is_directory=True)

# Default: no selection
assert config.can_select(file) == False
assert config.can_select(folder) == False

# With selection enabled
select_config = GalleryConfig(selection_mode=SelectionMode.SINGLE)
assert select_config.can_select(file) == True
assert select_config.can_select(folder) == False  # Folders never selectable

# Test with custom IDs
custom_config = GalleryConfig(
    gallery_id="my-gallery",
    content_id="my-content",
    grid=GridConfig(columns=6),
    pagination=PaginationConfig(items_per_page=48)
)
assert custom_config.gallery_id == "my-gallery"
assert custom_config.grid.columns == 6
assert custom_config.pagination.items_per_page == 48

print("GalleryConfig tests passed!")

GalleryConfig tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()